<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/03_seq2seq_lstm_french_to_ewe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Modèle Seq2Seq LSTM pour la traduction français → éwé

Dans ce notebook, nous construisons notre premier modèle de traduction automatique du français vers l’éwé.

Après avoir exploré et préparé le dataset dans les notebooks précédents, nous disposons maintenant de données numériques prêtes pour l'entraînement.

L’objectif est de construire un modèle Seq2Seq composé de deux grandes parties :

- un encodeur, qui lit la phrase française ;
- un décodeur, qui génère progressivement la phrase en éwé.

Ce premier modèle constitue notre baseline Deep Learning. Il nous permettra d’obtenir une première version fonctionnelle du système avant de tester des approches plus avancées.

In [3]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# On crée un dossier dédié au projet dans Google Drive.
# Tous les modèles et checkpoints importants seront sauvegardés ici.

import os

project_dir = "/content/drive/MyDrive/french_ewe_project"
models_dir = os.path.join(project_dir, "models")
artifacts_dir = os.path.join(project_dir, "artifacts")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(artifacts_dir, exist_ok=True)

print("Dossier projet :", project_dir)
print("Dossier modèles :", models_dir)
print("Dossier artifacts :", artifacts_dir)

Dossier projet : /content/drive/MyDrive/french_ewe_project
Dossier modèles : /content/drive/MyDrive/french_ewe_project/models
Dossier artifacts : /content/drive/MyDrive/french_ewe_project/artifacts


In [5]:
# On charge les fichiers générés pendant l'étape de prétraitement.
# Ces fichiers contiennent les phrases déjà transformées en séquences numériques,
# ainsi que les tokenizers et les paramètres importants du projet.

from google.colab import files

uploaded = files.upload()

Saving decoder_input_test_ewe.npy to decoder_input_test_ewe.npy
Saving decoder_input_test_fr.npy to decoder_input_test_fr.npy
Saving decoder_input_train_ewe.npy to decoder_input_train_ewe.npy
Saving decoder_input_train_fr.npy to decoder_input_train_fr.npy
Saving decoder_input_valid_ewe.npy to decoder_input_valid_ewe.npy
Saving decoder_input_valid_fr.npy to decoder_input_valid_fr.npy
Saving decoder_target_test_ewe.npy to decoder_target_test_ewe.npy
Saving decoder_target_test_fr.npy to decoder_target_test_fr.npy
Saving decoder_target_train_ewe.npy to decoder_target_train_ewe.npy
Saving decoder_target_train_fr.npy to decoder_target_train_fr.npy
Saving decoder_target_valid_ewe.npy to decoder_target_valid_ewe.npy
Saving decoder_target_valid_fr.npy to decoder_target_valid_fr.npy
Saving ewe_tokenizer.pkl to ewe_tokenizer.pkl
Saving french_tokenizer.pkl to french_tokenizer.pkl
Saving preprocessing_config.pkl to preprocessing_config.pkl
Saving X_test_ewe_pad.npy to X_test_ewe_pad.npy
Saving X_t

In [6]:
# On importe les bibliothèques nécessaires pour construire et entraîner le modèle.
#
# numpy permet de charger les tableaux numériques.
# pickle permet de récupérer les tokenizers sauvegardés.
# TensorFlow/Keras permet de construire le modèle Seq2Seq avec LSTM.

import numpy as np
import pickle
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [7]:
# On charge les données d'entraînement, de validation et de test.
#
# X correspond aux phrases françaises en entrée.
# decoder_input correspond aux phrases éwé données au décodeur pendant l'entraînement.
# decoder_target correspond aux phrases éwé que le modèle doit apprendre à prédire.

X_train_fr = np.load("X_train_fr_pad.npy")
decoder_input_train_ewe = np.load("decoder_input_train_ewe.npy")
decoder_target_train_ewe = np.load("decoder_target_train_ewe.npy")

X_valid_fr = np.load("X_valid_fr_pad.npy")
decoder_input_valid_ewe = np.load("decoder_input_valid_ewe.npy")
decoder_target_valid_ewe = np.load("decoder_target_valid_ewe.npy")

X_test_fr = np.load("X_test_fr_pad.npy")
decoder_input_test_ewe = np.load("decoder_input_test_ewe.npy")
decoder_target_test_ewe = np.load("decoder_target_test_ewe.npy")

In [8]:
# On vérifie les dimensions des tableaux.
# Cela permet de s'assurer que les entrées et les sorties sont bien alignées.

print("X_train_fr :", X_train_fr.shape)
print("decoder_input_train_ewe :", decoder_input_train_ewe.shape)
print("decoder_target_train_ewe :", decoder_target_train_ewe.shape)

print("\nX_valid_fr :", X_valid_fr.shape)
print("decoder_input_valid_ewe :", decoder_input_valid_ewe.shape)
print("decoder_target_valid_ewe :", decoder_target_valid_ewe.shape)

print("\nX_test_fr :", X_test_fr.shape)
print("decoder_input_test_ewe :", decoder_input_test_ewe.shape)
print("decoder_target_test_ewe :", decoder_target_test_ewe.shape)

X_train_fr : (18820, 50)
decoder_input_train_ewe : (18820, 49)
decoder_target_train_ewe : (18820, 49)

X_valid_fr : (2352, 50)
decoder_input_valid_ewe : (2352, 49)
decoder_target_valid_ewe : (2352, 49)

X_test_fr : (2353, 50)
decoder_input_test_ewe : (2353, 49)
decoder_target_test_ewe : (2353, 49)


In [9]:
# On recharge les tokenizers créés dans le notebook précédent.
# Ils seront utiles pour comprendre les indices des mots et, plus tard,
# pour reconstruire une phrase traduite à partir des prédictions du modèle.

with open("french_tokenizer.pkl", "rb") as f:
    french_tokenizer = pickle.load(f)

with open("ewe_tokenizer.pkl", "rb") as f:
    ewe_tokenizer = pickle.load(f)

with open("preprocessing_config.pkl", "rb") as f:
    config = pickle.load(f)

config

{'VOCAB_SIZE': 10000,
 'french_vocab_size': 10000,
 'ewe_vocab_size': 10000,
 'MAX_LEN_FRENCH': 50,
 'MAX_LEN_EWE': 50}

In [10]:
# On récupère les paramètres importants depuis le fichier de configuration.
# Ces valeurs doivent être les mêmes que celles utilisées pendant le prétraitement.

french_vocab_size = config["french_vocab_size"]
ewe_vocab_size = config["ewe_vocab_size"]

MAX_LEN_FRENCH = config["MAX_LEN_FRENCH"]
MAX_LEN_EWE = config["MAX_LEN_EWE"]

print("Vocabulaire français :", french_vocab_size)
print("Vocabulaire éwé :", ewe_vocab_size)
print("Longueur max français :", MAX_LEN_FRENCH)
print("Longueur max éwé :", MAX_LEN_EWE)

Vocabulaire français : 10000
Vocabulaire éwé : 10000
Longueur max français : 50
Longueur max éwé : 50


In [11]:
# Le modèle va prédire, pour chaque position de la phrase, le mot éwé le plus probable.
#
# Comme on utilise sparse_categorical_crossentropy, les sorties attendues doivent être
# des entiers, mais avec une dimension finale supplémentaire.
#
# Exemple :
# avant  : (nombre_phrases, longueur_phrase)
# après  : (nombre_phrases, longueur_phrase, 1)

# Le modèle attend une cible avec une dimension finale supplémentaire.
# Mais on vérifie d'abord la forme actuelle pour éviter d'ajouter plusieurs fois
# la même dimension si la cellule est relancée.

def ensure_target_shape(target):
    if len(target.shape) == 2:
        target = np.expand_dims(target, -1)
    return target

decoder_target_train_ewe = ensure_target_shape(decoder_target_train_ewe)
decoder_target_valid_ewe = ensure_target_shape(decoder_target_valid_ewe)
decoder_target_test_ewe = ensure_target_shape(decoder_target_test_ewe)

print("Nouvelle forme decoder_target_train_ewe :", decoder_target_train_ewe.shape)
print("Nouvelle forme decoder_target_valid_ewe :", decoder_target_valid_ewe.shape)
print("Nouvelle forme decoder_target_test_ewe :", decoder_target_test_ewe.shape)

Nouvelle forme decoder_target_train_ewe : (18820, 49, 1)
Nouvelle forme decoder_target_valid_ewe : (2352, 49, 1)
Nouvelle forme decoder_target_test_ewe : (2353, 49, 1)


## Construction du modèle Seq2Seq

Le modèle utilisé ici est basé sur une architecture Encodeur-Décodeur.

L’encodeur reçoit la phrase française et la transforme en un état interne.  
Cet état représente une sorte de résumé numérique de la phrase.

Le décodeur utilise ensuite cet état pour générer la phrase en éwé mot par mot.

Cette architecture est classique pour les premiers systèmes de traduction automatique neuronale.

In [12]:
# On définit quelques hyperparamètres du modèle.
#
# EMBEDDING_DIM correspond à la taille des vecteurs qui représenteront les mots.
# LATENT_DIM correspond à la taille de la mémoire interne du LSTM.
#
# Pour une première version, on choisit des valeurs raisonnables afin que le modèle
# puisse s'entraîner sur Google Colab sans être trop lourd.

EMBEDDING_DIM = 128
LATENT_DIM = 128
DROPOUT_RATE = 0.3

In [13]:
# -------------------------
# Encodeur
# -------------------------

# L'encodeur reçoit une phrase française déjà transformée en séquence d'entiers.
encoder_inputs = Input(shape=(MAX_LEN_FRENCH,), name="encoder_inputs")

# La couche Embedding transforme chaque entier en vecteur dense.
# mask_zero=True permet d'ignorer les zéros ajoutés par le padding.
encoder_embedding = Embedding(
    input_dim=french_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

# Le LSTM lit la phrase française et retourne son état final.
# Ces états seront transmis au décodeur.
encoder_lstm = LSTM(
    LATENT_DIM,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# On garde les états finaux de l'encodeur.
encoder_states = [state_h, state_c]


# -------------------------
# Décodeur
# -------------------------

# Le décodeur reçoit la phrase cible décalée.
# Pendant l'entraînement, on lui donne le début de la traduction correcte.
decoder_inputs = Input(shape=(MAX_LEN_EWE - 1,), name="decoder_inputs")

decoder_embedding = Embedding(
    input_dim=ewe_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding"
)(decoder_inputs)

# Le décodeur LSTM reçoit comme état initial les états produits par l'encodeur.
decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# On ajoute un Dropout pour réduire le risque de surapprentissage.
decoder_outputs = Dropout(DROPOUT_RATE)(decoder_outputs)

# La couche Dense prédit le mot suivant parmi tout le vocabulaire éwé.
decoder_dense = Dense(
    ewe_vocab_size,
    activation="softmax",
    name="output_dense"
)

decoder_outputs = decoder_dense(decoder_outputs)


# -------------------------
# Modèle complet
# -------------------------

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name="seq2seq_lstm_french_to_ewe"
)

In [14]:
# On compile le modèle.
#
# La loss sparse_categorical_crossentropy est adaptée ici car nos mots cibles
# sont représentés par des entiers et non par des vecteurs one-hot.
#
# L'optimizer Adam est un choix standard pour commencer.

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "seq2seq_lstm_french_to_ewe"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 50, 128)   │  1,280,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 49, 128)   │  1,280,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    131,584 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 49, 128), │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 49, 128)   │          0 │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 49, 10000) │  1,290,000 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,113,168 (15.69 MB)

 Trainable params: 4,113,168 (15.69 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Les callbacks permettent de mieux contrôler l'entraînement.
#
# EarlyStopping arrête l'entraînement si la validation ne s'améliore plus.
# ModelCheckpoint sauvegarde automatiquement le meilleur modèle obtenu.

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Cette fois, le meilleur modèle sera sauvegardé directement dans Google Drive.
# Même si Colab se déconnecte, ce fichier restera disponible.

checkpoint_path = os.path.join(models_dir, "best_seq2seq_fr_to_ewe.keras")

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [16]:
# On entraîne maintenant le modèle.
#
# Le modèle reçoit deux entrées :
# - la phrase française pour l'encodeur ;
# - la phrase éwé décalée pour le décodeur.
#
# La sortie attendue est la phrase éwé également décalée, c'est-à-dire
# les mots que le modèle doit prédire à chaque étape.

history = model.fit(
    [X_train_fr, decoder_input_train_ewe],
    decoder_target_train_ewe,

    validation_data=(
        [X_valid_fr, decoder_input_valid_ewe],
        decoder_target_valid_ewe
    ),

    epochs=20,
    batch_size=64,
    callbacks=[early_stopping, checkpoint]
)

Epoch 1/20
295/295 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0979 - loss: 6.9064
Epoch 1: val_loss improved from None to 5.46199, saving model to /content/drive/MyDrive/french_ewe_project/models/best_seq2seq_fr_to_ewe.keras

Epoch 1: finished saving model to /content/drive/MyDrive/french_ewe_project/models/best_seq2seq_fr_to_ewe.keras
295/295 ━━━━━━━━━━━━━━━━━━━━ 655s 2s/step - accuracy: 0.1277 - loss: 6.1986 - val_accuracy: 0.1991 - val_loss: 5.4620
Epoch 2/20
295/295 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1795 - loss: 5.6304
Epoch 2: val_loss improved from 5.46199 to 5.28819, saving model to /content/drive/MyDrive/french_ewe_project/models/best_seq2seq_fr_to_ewe.keras

Epoch 2: finished saving model to /content/drive/MyDrive/french_ewe_project/models/best_seq2seq_fr_to_ewe.keras
295/295 ━━━━━━━━━━━━━━━━━━━━ 660s 2s/step - accuracy: 0.1826 - loss: 5.6012 - val_accuracy: 0.2087 - val_loss: 5.2882
Epoch 3/20
295/295 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1900 - loss: 5

In [17]:
# On sauvegarde aussi le modèle final dans Google Drive.

final_model_path = os.path.join(models_dir, "seq2seq_lstm_fr_to_ewe_final.keras")

model.save(final_model_path)

print("Modèle final sauvegardé ici :", final_model_path)
print("Meilleur modèle sauvegardé ici :", checkpoint_path)

Modèle final sauvegardé ici : /content/drive/MyDrive/french_ewe_project/models/seq2seq_lstm_fr_to_ewe_final.keras
Meilleur modèle sauvegardé ici : /content/drive/MyDrive/french_ewe_project/models/best_seq2seq_fr_to_ewe.keras


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

models_dir = "/content/drive/MyDrive/french_ewe_project/models"

if os.path.exists(models_dir):
    print(os.listdir(models_dir))
else:
    print("Le dossier models n'existe pas encore.")

['best_seq2seq_fr_to_ewe.keras', 'seq2seq_lstm_fr_to_ewe_final.keras']


In [4]:
import tensorflow as tf
import os

models_dir = "/content/drive/MyDrive/french_ewe_project/models"
checkpoint_path = os.path.join(models_dir, "best_seq2seq_fr_to_ewe.keras")

model = tf.keras.models.load_model(checkpoint_path)

print("Meilleur modèle rechargé avec succès.")
model.summary()

Meilleur modèle rechargé avec succès.


Model: "seq2seq_lstm_french_to_ewe"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 50, 128)   │  1,280,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 49, 128)   │  1,280,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    131,584 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 49, 128), │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 49, 128)   │          0 │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 49, 10000) │  1,290,000 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,339,506 (47.07 MB)

 Trainable params: 4,113,168 (15.69 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,226,338 (31.38 MB)

## Rechargement du meilleur modèle

Après l'entraînement, le meilleur modèle a été sauvegardé automatiquement dans Google Drive grâce au callback `ModelCheckpoint`.

Cela permet de sécuriser l'expérience : même si le runtime Colab se déconnecte, le modèle entraîné reste disponible dans Drive.

Nous rechargeons ici le meilleur modèle sauvegardé afin de l'évaluer sur le jeu de test.

In [6]:
from google.colab import files

uploaded = files.upload()

Saving decoder_target_test_ewe.npy to decoder_target_test_ewe.npy
Saving X_test_fr_pad.npy to X_test_fr_pad.npy
Saving decoder_input_test_ewe.npy to decoder_input_test_ewe.npy


In [7]:
import numpy as np

X_test_fr = np.load("X_test_fr_pad.npy")
decoder_input_test_ewe = np.load("decoder_input_test_ewe.npy")
decoder_target_test_ewe = np.load("decoder_target_test_ewe.npy")

In [8]:
def ensure_target_shape(target):
    if len(target.shape) == 2:
        target = np.expand_dims(target, -1)
    return target

decoder_target_test_ewe = ensure_target_shape(decoder_target_test_ewe)

print("X_test_fr :", X_test_fr.shape)
print("decoder_input_test_ewe :", decoder_input_test_ewe.shape)
print("decoder_target_test_ewe :", decoder_target_test_ewe.shape)

X_test_fr : (2353, 50)
decoder_input_test_ewe : (2353, 49)
decoder_target_test_ewe : (2353, 49, 1)


In [9]:
test_loss, test_accuracy = model.evaluate(
    [X_test_fr, decoder_input_test_ewe],
    decoder_target_test_ewe
)

print("Loss sur le test set :", test_loss)
print("Accuracy sur le test set :", test_accuracy)

74/74 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.2576 - loss: 4.7163
Loss sur le test set : 4.716343879699707
Accuracy sur le test set : 0.2576228678226471


## Évaluation du modèle

Le modèle Seq2Seq LSTM français → éwé a été évalué sur le jeu de test, constitué de phrases jamais vues pendant l'entraînement.

Les résultats obtenus sont les suivants :

- Loss sur le test set : 4.7163
- Accuracy sur le test set : 0.2576

L'accuracy obtenue correspond à une accuracy mot par mot. Elle ne doit donc pas être interprétée comme une mesure parfaite de la qualité globale de traduction, contrairement à une tâche de classification classique.

Ces résultats montrent que le modèle apprend certaines correspondances entre les phrases françaises et les phrases en éwé, mais ses performances restent limitées. Cela peut s'expliquer par la difficulté de la traduction automatique, la taille relativement modeste du corpus, l'absence de mécanisme d'attention et le fait que l’éwé est une langue faiblement représentée dans les ressources NLP.

Ce modèle constitue donc une première baseline Deep Learning. Il servira de point de comparaison pour des améliorations futures, comme l'ajout d'un mécanisme d'attention, l'utilisation d'un Transformer ou le fine-tuning d'un modèle multilingue pré-entraîné.

# Test de traduction avec le modèle entraîné

Après l'entraînement et l'évaluation du modèle, nous allons maintenant tester sa capacité à produire une traduction en éwé à partir d'une phrase française.

Pendant l'entraînement, le modèle reçoit la phrase française ainsi qu'une partie de la phrase éwé correcte.  
Mais pendant l'inférence, c'est différent : le modèle doit générer la traduction mot par mot, en commençant par le token `<start>` jusqu'au token `<end>`.

Pour réaliser ce test, nous utilisons le modèle entraîné pour générer progressivement une séquence en éwé.

La phrase française est d'abord nettoyée, tokenisée et transformée en séquence numérique. Ensuite, on initialise la séquence du décodeur avec le token `<start>`, puis le modèle prédit les mots suivants un par un jusqu’à atteindre le token `<end>` ou la longueur maximale autorisée.

Cette méthode permet d'obtenir une première démonstration de traduction, même si elle reste limitée par l’architecture Seq2Seq LSTM simple utilisée ici.

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf

models_dir = "/content/drive/MyDrive/french_ewe_project/models"
checkpoint_path = os.path.join(models_dir, "best_seq2seq_fr_to_ewe.keras")

print("Dossier models :", os.listdir(models_dir))

model = tf.keras.models.load_model(checkpoint_path)

print("Modèle rechargé avec succès.")
print("Nom du modèle :", model.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier models : ['best_seq2seq_fr_to_ewe.keras', 'seq2seq_lstm_fr_to_ewe_final.keras']
Modèle rechargé avec succès.
Nom du modèle : seq2seq_lstm_french_to_ewe


In [2]:
from google.colab import files
uploaded = files.upload()

Saving ewe_tokenizer.pkl to ewe_tokenizer.pkl
Saving french_tokenizer.pkl to french_tokenizer.pkl


In [17]:
import pickle

with open("french_tokenizer.pkl", "rb") as f:
    french_tokenizer = pickle.load(f)

with open("ewe_tokenizer.pkl", "rb") as f:
    ewe_tokenizer = pickle.load(f)

with open("preprocessing_config.pkl", "rb") as f:
    config = pickle.load(f)

MAX_LEN_FRENCH = config["MAX_LEN_FRENCH"]
MAX_LEN_EWE = config["MAX_LEN_EWE"]

print("Tokenizers et configuration rechargés.")
print("MAX_LEN_FRENCH :", MAX_LEN_FRENCH)
print("MAX_LEN_EWE :", MAX_LEN_EWE)

Tokenizers et configuration rechargés.
MAX_LEN_FRENCH : 50
MAX_LEN_EWE : 50


In [18]:
# Le tokenizer transforme les mots en indices.
# Pour générer une phrase lisible, on doit faire l'opération inverse :
# transformer les indices prédits par le modèle en mots.

reverse_ewe_word_index = {
    index: word for word, index in ewe_tokenizer.word_index.items()
}

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def translate_fr_to_ewe(sentence):
    # On nettoie la phrase française comme pendant le prétraitement.
    cleaned_sentence = clean_text(sentence)

    # On transforme la phrase en séquence numérique avec le tokenizer français.
    input_sequence = french_tokenizer.texts_to_sequences([cleaned_sentence])

    # On applique le padding pour obtenir la même longueur que pendant l'entraînement.
    input_sequence = pad_sequences(
        input_sequence,
        maxlen=MAX_LEN_FRENCH,
        padding="post",
        truncating="post"
    )

    # On récupère le token de début de phrase côté éwé.
    start_token = ewe_tokenizer.word_index.get("<start>")

    if start_token is None:
        return "Erreur : le token <start> n'existe pas dans le tokenizer éwé."

    # Le décodeur commence avec uniquement le token <start>.
    decoder_sequence = np.zeros((1, MAX_LEN_EWE - 1))
    decoder_sequence[0, 0] = start_token

    translated_words = []

    # Le modèle génère la traduction mot par mot.
    for i in range(1, MAX_LEN_EWE - 1):
        predictions = model.predict(
            [input_sequence, decoder_sequence],
            verbose=0
        )

        predicted_id = np.argmax(predictions[0, i - 1, :])
        predicted_word = reverse_ewe_word_index.get(predicted_id, "")

        if predicted_word == "<end>":
            break

        if predicted_word not in ["", "<start>", "<unk>"]:
            translated_words.append(predicted_word)

        decoder_sequence[0, i] = predicted_id

    return " ".join(translated_words)

In [20]:
phrases_test = [
    "bonjour",
    "comment tu vas ?",
    "je suis malade",
    "je veux boire de l'eau",
    "merci beaucoup",
    "je vais à la maison"
]

for phrase in phrases_test:
    traduction = translate_fr_to_ewe(phrase)
    print("Français :", phrase)
    print("Éwé      :", traduction)
    print("-" * 80)

Français : bonjour
Éwé      : 
--------------------------------------------------------------------------------
Français : comment tu vas ?
Éwé      : 
--------------------------------------------------------------------------------
Français : je suis malade
Éwé      : 
--------------------------------------------------------------------------------
Français : je veux boire de l'eau
Éwé      : nye nye
--------------------------------------------------------------------------------
Français : merci beaucoup
Éwé      : 
--------------------------------------------------------------------------------
Français : je vais à la maison
Éwé      : 
--------------------------------------------------------------------------------


## Analyse des traductions générées

Les premiers tests montrent que le modèle est capable de produire certaines sorties en éwé, mais les traductions restent très limitées. Pour plusieurs phrases simples, le modèle génère une sortie vide ou des mots répétés comme `nye nye`.

Ce comportement est cohérent avec les résultats quantitatifs obtenus précédemment. L’accuracy mot par mot sur le test set est d’environ 25,76 %, ce qui indique que le modèle apprend certaines régularités du corpus, mais ne maîtrise pas encore suffisamment la génération complète de phrases.

Ces limites peuvent s’expliquer par plusieurs facteurs :

- le modèle Seq2Seq LSTM utilisé est simple ;
- il ne possède pas de mécanisme d’attention ;
- le vocabulaire est relativement large ;
- l’éwé est une langue peu représentée dans les ressources NLP ;
- certaines phrases courtes de test peuvent être absentes ou rares dans le corpus d’entraînement.

Cette étape reste donc importante, car elle constitue une première baseline fonctionnelle. Elle montre que la chaîne complète fonctionne : chargement des données, entraînement, sauvegarde du modèle, évaluation et génération de traductions.

Pour améliorer les résultats, les prochaines étapes possibles sont l’ajout d’un mécanisme d’attention, l’utilisation d’un Transformer ou le fine-tuning d’un modèle multilingue pré-entraîné.